# Entrenamiento y métricas de validacion - RF-DETR

En este notebook se documenta el entrenamiento de RF-DETR-Small con el dataset de parches (original y denoised), y el cálculo de las métricas sobre el conjunto de test.

## Entrenamiento - Parches Original

Se entrenó RF-DETR-Small con el dataset de parches original (ejecutado en tmux, sesión "rfdetr"). Se usaron los mismos parámetros generales que YOLO y RT-DETR donde fue 
posible (epochs=50, patience=10, seed=42, batch=2). La resolución se dejó en el valor por defecto del modelo (672px), ya que con 1024px (la misma usada en YOLO/RT-DETR) se 
produjo un error de memoria GPU, incluso con batch=1.

Código ejecutado:

In [ ]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()

model.train(
    dataset_dir="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales",
    epochs=50,
    batch_size=2,
    grad_accum_steps=4,
    lr=1e-4,
    early_stopping_patience=10,
    seed=42,
    output_dir="/home/salvarado/TFM/resultados/result_RF_DTR/parches_original_50ep_RF_DTR"
)

### Resultado del entrenamiento - Parches Original (última época, val)

| Métrica | Valor |
|---|---|
| Precisión | 0.9338 |
| Recall | 0.9322 |
| mAP50 | 0.9493 |
| mAP50-95 | 0.6220 |

Completó las 50 épocas (sin early stopping). Mejor checkpoint guardado según mAP 
regular: 0.6554 (EMA: 0.6490).

### Análisis

- Precisión (0.9338) y recall (0.9322) muy equilibrados y altos → el modelo detecta 
  bien los kanjis sin sacrificar uno por el otro.

- F1 alto (0.9330) → buen balance general entre precisión y recall.

- AP 50:95 (0.6220) notablemente más bajo que el F1 → el modelo encuentra bien los 
  kanjis, pero el ajuste preciso de las cajas (a niveles de exigencia altos) tiene 
  margen de mejora.

- Entrenó las 50 épocas completas sin activar early stopping → el modelo seguía 
  mejorando de forma gradual hasta el final del entrenamiento.

## Entrenamiento - Parches Denoised

Se repitió el mismo proceso con el dataset de parches denoised (ejecutado en tmux, sesión "rfdetr2"), usando los mismos parámetros que en la versión original.

Código ejecutado:

In [ ]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()

model.train(
    dataset_dir="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised",
    epochs=50,
    batch_size=2,
    grad_accum_steps=4,
    lr=1e-4,
    early_stopping_patience=10,
    seed=42,
    output_dir="/home/salvarado/TFM/resultados/result_RF_DTR/parches_denoised_50ep_RF_DTR"
)

### Resultado del entrenamiento - Parches Denoised (última época, val)

| Métrica | Valor |
|---|---|
| Precisión | 0.9314 |
| Recall | 0.9163 |
| mAP50 | 0.9464 |
| mAP50-95 | 0.6234 |

Completó las 50 épocas (sin early stopping). Mejor checkpoint guardado según mAP 
regular: 0.6455 (EMA: 0.6414).

### Análisis

- Entrenó las 50 épocas completas sin activar early stopping, igual que la versión 
  original.

- Precisión (0.9314) y recall (0.9163) altos, con un comportamiento equilibrado 
  entre ambos.

- AP 50:95 (0.6234) es bastante más bajo que el F1 (0.9238) → el modelo encuentra 
  los kanjis bien, pero no siempre dibuja la caja perfectamente ajustada. Pasa lo 
  mismo que en la versión original.

## Cálculo de métricas en test parches original - RF-DETR

A diferencia de YOLO y RT-DETR (Ultralytics), RF-DETR no ofrece un comando directo para evaluar el modelo sobre el conjunto de test. Solo calcula métricas automáticas 
sobre el conjunto de validación durante el entrenamiento.Para obtener métricas de test comparables, se utilizó la librería `supervision` (de 
Roboflow, los mismos creadores de RF-DETR), que permite cargar un dataset y calcular métricas estándar conectándolo con cualquier modelo.

In [1]:
import supervision as sv
# Cargar el dataset de test (mismas imágenes/labels usadas para YOLO y RT-DETR)
dataset = sv.DetectionDataset.from_yolo(
    images_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales/images/test",
    annotations_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales/labels/test",
    data_yaml_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales/dataset_parches_originales.yaml"
)

print(f"Número de imágenes en el dataset: {len(dataset)}")
print(f"Clases: {dataset.classes}")

Número de imágenes en el dataset: 240
Clases: ['kanji']


In [2]:
from rfdetr import RFDETRSmall

# Cargar el modelo entrenado (checkpoint final de RF-DETR)
model = RFDETRSmall(pretrain_weights="/home/salvarado/TFM/resultados/result_RF_DTR/parches_original_50ep_RF_DTR/checkpoint_best_total.pth")

print("Modelo cargado correctamente")

[2026-06-17 03:22:31] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-17 03:22:31] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-17 03:22:32] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
[2026-06-17 03:22:32] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Modelo cargado correctamente


In [3]:
import supervision as sv

# Conectar el modelo con supervision
def callback(image):
    detections = model.predict(image)
    return detections

# Calcular mAP sobre el conjunto de test
map_metric = sv.MeanAveragePrecision.benchmark(dataset=dataset, callback=callback)

print(f"mAP50:    {map_metric.map50:.3f}")
print(f"mAP50-95: {map_metric.map50_95:.3f}")

[2026-06-17 03:22:36] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


mAP50:    0.961
mAP50-95: 0.681


## Cálculo de métricas en test - RF-DETR Parches Denoised

Mismo proceso que para la versión original: librería supervision, dataset de test original, checkpoint_best_total.pth del modelo denoised.

In [ ]:
import supervision as sv
from rfdetr import RFDETRSmall

# Cargar el dataset de test denoised (dataset original, no la copia reorganizada)
dataset_denoised = sv.DetectionDataset.from_yolo(
    images_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/denoised/images/test",
    annotations_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/denoised/labels/test",
    data_yaml_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/denoised/dataset_parches_denoised.yaml"
)

# Cargar el modelo entrenado con denoised
model_denoised = RFDETRSmall(pretrain_weights="/home/salvarado/TFM/resultados/result_RF_DTR/parches_denoised_50ep_RF_DTR/checkpoint_best_total.pth")

print(f"Número de imágenes: {len(dataset_denoised)}")

[2026-06-17 01:05:15] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-17 01:05:15] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-17 01:05:16] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
[2026-06-17 01:05:16] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Número de imágenes: 240


In [ ]:
def callback_denoised(image):
    return model_denoised.predict(image)

map_metric_denoised = sv.MeanAveragePrecision.benchmark(dataset=dataset_denoised, callback=callback_denoised)

print(f"mAP50:    {map_metric_denoised.map50:.3f}")
print(f"mAP50-95: {map_metric_denoised.map50_95:.3f}")

[2026-06-17 01:05:39] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


mAP50:    0.960
mAP50-95: 0.703


### Cargar dataset y modelo (RF-DETR Parches Original)

In [2]:
import supervision as sv
from rfdetr import RFDETRSmall

# Cargar dataset de test
dataset = sv.DetectionDataset.from_yolo(
    images_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales/images/test",
    annotations_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales/labels/test",
    data_yaml_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales/dataset_parches_originales.yaml"
)

# Cargar modelo entrenado
model = RFDETRSmall(pretrain_weights="/home/salvarado/TFM/resultados/result_RF_DTR/parches_original_50ep_RF_DTR/checkpoint_best_total.pth")

print(f"Dataset: {len(dataset)} imágenes, Clases: {dataset.classes}")

[2026-06-23 17:59:11] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-23 17:59:11] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-23 17:59:11] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
[2026-06-23 17:59:12] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Dataset: 240 imágenes, Clases: ['kanji']


### Paso 2: Calcular Precisión y Recall (Original)

Para obtener Precisión y Recall, se usa la matriz de confusión de supervision. 
Esta matriz  da los valores de TP (verdaderos positivos), FP (falsos positivos) 
y FN (falsos negativos), a partir de los cuales se calcula las métricas manualmente.

In [4]:
confusion_matrix = sv.ConfusionMatrix.benchmark(
    dataset=dataset,
    callback=callback
)

print("Matriz:")
print(confusion_matrix.matrix)
print("Clases:", confusion_matrix.classes)

Matriz:
[[3450.  207.]
 [ 211.    0.]]
Clases: ['kanji']


In [5]:
TP = confusion_matrix.matrix[0, 0]
FN = confusion_matrix.matrix[0, 1]
FP = confusion_matrix.matrix[1, 0]

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print(f"Precisión: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

Precisión: 0.942
Recall:    0.943


### Análisis - Precisión y Recall RF-DETR Parches Original (test)

- Precisión (0.942) y Recall (0.943) muy equilibrados → el modelo detecta bien 
  los kanjis sin sacrificar uno por el otro.

- Tanto Precisión como Recall están muy cerca del 94%, lo que indica un 
  comportamiento consistente y fiable del modelo sobre el conjunto de test.

### Cargar dataset y modelo (RF-DETR Parches Denoised)

In [1]:
import supervision as sv
from rfdetr import RFDETRSmall

# Cargar dataset de test
dataset_denoised = sv.DetectionDataset.from_yolo(
    images_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/denoised/images/test",
    annotations_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/denoised/labels/test",
    data_yaml_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/denoised/dataset_parches_denoised.yaml"
)

# Cargar modelo entrenado
model_denoised = RFDETRSmall(pretrain_weights="/home/salvarado/TFM/resultados/result_RF_DTR/parches_denoised_50ep_RF_DTR/checkpoint_best_total.pth")

print(f"Dataset: {len(dataset_denoised)} imágenes, Clases: {dataset_denoised.classes}")

[2026-06-23 19:15:38] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-23 19:15:38] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-23 19:15:39] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
[2026-06-23 19:15:39] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Dataset: 240 imágenes, Clases: ['kanji']


### Paso 2: Calcular Precisión y Recall (Denoised)

In [2]:
def callback_denoised(image):
    return model_denoised.predict(image)

confusion_matrix_denoised = sv.ConfusionMatrix.benchmark(
    dataset=dataset_denoised,
    callback=callback_denoised
)

matrix = confusion_matrix_denoised.matrix
TP = matrix[0, 0]
FN = matrix[0, 1]
FP = matrix[1, 0]

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print(f"Precisión: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

[2026-06-23 19:16:32] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Precisión: 0.935
Recall:    0.944


### Análisis - Precisión y Recall RF-DETR Parches Denoised (test)

- Precisión (0.935) y Recall (0.944) muy equilibrados, igual que en la versión original  el modelo mantiene un comportamiento consistente independientemente 
  del preprocesado.

- Las diferencias respecto a la versión original son mínimas (0.007 en Precisión, 0.001 en Recall) →el denoising no tiene un impacto significativo en RF-DETR 
  con parches, igual que ocurrió con YOLO.

## Cálculo de métricas en test - RF-DETR Imagen Completa Original

Mismo proceso que para parches: usamos supervision para cargar el dataset de test y calcular mAP50, mAP50-95, Precisión y Recall. El modelo fue entrenado en Google Colab (GPU A100, 40GB).

### Paso 1: Cargar dataset y modelo

In [1]:
import supervision as sv
from rfdetr import RFDETRSmall

dataset_completa_original = sv.DetectionDataset.from_yolo(
    images_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_prep/dataset_original/images/test",
    annotations_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_prep/dataset_original/labels/test",
    data_yaml_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_prep/dataset_original/dataset_original.yaml"
)

model_completa_original = RFDETRSmall(pretrain_weights="/home/salvarado/TFM/resultados/result_RF_DTR/imagen_completa_original_50ep_RF_DTR/checkpoint_best_total.pth")

print(f"Dataset: {len(dataset_completa_original)} imágenes, Clases: {dataset_completa_original.classes}")

[2026-06-24 01:09:14] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-24 01:09:14] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-24 01:09:15] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
[2026-06-24 01:09:15] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.
[2026-06-24 01:09:15] [WARNING] rf-detr - Pretrained weights at '/home/salvarado/TFM/resultados/result_RF_DTR/imagen_completa_original_50ep_RF_DTR/checkpoint_best_total.pth' loaded only partially — this typically produces lower accuracy. 1

Dataset: 10 imágenes, Clases: ['kanji']


### Paso 2: Calcular mAP - RF-DETR Imagen Completa Original (test)

In [2]:
def callback_completa_original(image):
    return model_completa_original.predict(image)

map_metric_completa_original = sv.MeanAveragePrecision.benchmark(
    dataset=dataset_completa_original,
    callback=callback_completa_original
)

print(f"mAP50:    {map_metric_completa_original.map50:.3f}")
print(f"mAP50-95: {map_metric_completa_original.map50_95:.3f}")

libpng warning: iCCP: known incorrect sRGB profile
[2026-06-24 01:10:34] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


mAP50:    0.886
mAP50-95: 0.577


### Paso 3: Calcular Precisión y Recall - RF-DETR Imagen Completa Original (test)

In [3]:
confusion_matrix_completa_original = sv.ConfusionMatrix.benchmark(
    dataset=dataset_completa_original,
    callback=callback_completa_original
)

matrix = confusion_matrix_completa_original.matrix
TP = matrix[0, 0]
FN = matrix[0, 1]
FP = matrix[1, 0]

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print(f"Precisión: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


Precisión: 0.924
Recall:    0.818


### Cargar dataset y modelo - RF-DETR Imagen Completa Denoised (test)

In [4]:
dataset_completa_denoised = sv.DetectionDataset.from_yolo(
    images_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_prep/dataset_denoised/images/test",
    annotations_directory_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_prep/dataset_denoised/labels/test",
    data_yaml_path="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_prep/dataset_denoised/dataset_denoised.yaml"
)

model_completa_denoised = RFDETRSmall(pretrain_weights="/home/salvarado/TFM/resultados/result_RF_DTR/imagen_completa_denoised_50ep_RF_DTR/checkpoint_best_total.pth")

print(f"Dataset: {len(dataset_completa_denoised)} imágenes, Clases: {dataset_completa_denoised.classes}")

[2026-06-24 01:16:39] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-24 01:16:39] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-24 01:16:39] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
[2026-06-24 01:16:39] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.
[2026-06-24 01:16:39] [WARNING] rf-detr - Pretrained weights at '/home/salvarado/TFM/resultados/result_RF_DTR/imagen_completa_denoised_50ep_RF_DTR/checkpoint_best_total.pth' loaded only partially — this typically produces lower accuracy. 1

Dataset: 10 imágenes, Clases: ['kanji']


### Calcular métricas - RF-DETR Imagen Completa Denoised (test)

In [5]:
def callback_completa_denoised(image):
    return model_completa_denoised.predict(image)

map_metric_completa_denoised = sv.MeanAveragePrecision.benchmark(
    dataset=dataset_completa_denoised,
    callback=callback_completa_denoised
)

print(f"mAP50:    {map_metric_completa_denoised.map50:.3f}")
print(f"mAP50-95: {map_metric_completa_denoised.map50_95:.3f}")

[2026-06-24 01:17:55] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


mAP50:    0.897
mAP50-95: 0.583


### Paso 3: Calcular Precisión y Recall - RF-DETR Imagen Completa Denoised (test)

In [6]:
confusion_matrix_completa_denoised = sv.ConfusionMatrix.benchmark(
    dataset=dataset_completa_denoised,
    callback=callback_completa_denoised
)

matrix = confusion_matrix_completa_denoised.matrix
TP = matrix[0, 0]
FN = matrix[0, 1]
FP = matrix[1, 0]

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print(f"Precisión: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

Precisión: 0.953
Recall:    0.816
